# Exportação do MobileViTV2 para LiteRT/TFLite

In [11]:
from pathlib import Path

import litert_torch
import numpy as np
import torch
import torch.nn as nn
from ai_edge_litert.interpreter import Interpreter
from ai_edge_quantizer import quantizer, recipe
from sklearn.metrics import accuracy_score, classification_report
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import MobileViTV2Config, MobileViTV2ForImageClassification
from transformers.models.mobilevitv2.modeling_mobilevitv2 import (
    MobileViTV2Layer,
    MobileViTV2LinearSelfAttention,
)

In [26]:
DATASET_PATH = Path("/home/jovyan/work/UFSC/butterflies_austria/Dataset_train_test_split/butterflies-austria")
CHECKPOINT_PATH = Path("best_mobilevit_v3_small.pt")
EXPORT_DIR = Path("tflite_models")
EXPORT_DIR.mkdir(exist_ok=True)

MODEL_NAME = "apple/mobilevitv2-1.0-imagenet1k-256"
IMG_SIZE = 256
RESIZE_SIZE = 288
BATCH_SIZE = 64
NUM_CALIB_SAMPLES = 100

FP32_PATH = EXPORT_DIR / "model_fp32.tflite"
INT8A16_PATH = EXPORT_DIR / "model_int8a16.tflite"
INT8_PATH = EXPORT_DIR / "model_int8.tflite"

## Dataset

In [27]:
image_transform = transforms.Compose([
    transforms.Resize(RESIZE_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.Lambda(lambda image: image.convert("RGB")),
    transforms.ToTensor(),
    transforms.Lambda(lambda image: image[[2, 1, 0]]),
])

calibration_dataset = datasets.ImageFolder(DATASET_PATH / "train", transform=image_transform)
test_dataset = datasets.ImageFolder(DATASET_PATH / "test", transform=image_transform)

if calibration_dataset.class_to_idx != test_dataset.class_to_idx:
    raise ValueError("Training and test classes are not aligned")

class_names = calibration_dataset.classes
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=4,
    persistent_workers=True,
)

print(f"Classes: {len(class_names)}")
print(f"Calibration: {len(calibration_dataset)} | Test: {len(test_dataset)}")

Classes: 20
Calibration: 5183 | Test: 648


## Modelo e checkpoint

In [14]:
class MobileViTClassifier(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        return self.model(pixel_values=x, return_dict=False)[0]


config = MobileViTV2Config.from_pretrained(
    MODEL_NAME,
    num_labels=len(class_names),
    id2label={index: name for index, name in enumerate(class_names)},
    label2id={name: index for index, name in enumerate(class_names)},
    classifier_dropout_prob=0.3,
)
model = MobileViTClassifier(MobileViTV2ForImageClassification(config))
state_dict = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=True)
model.load_state_dict(state_dict, strict=True)
model.eval()

print(f"Checkpoint carregado: {CHECKPOINT_PATH}")
print(f"Parâmetros: {sum(parameter.numel() for parameter in model.parameters()):,}")

[transformers] You passed `num_labels=20` which is incompatible to the `id2label` map of length `1000`.


Checkpoint carregado: best_mobilevit_v3_small.pt
Parâmetros: 4,399,101


## Atenção compatível com LiteRT

In [15]:
class LiteRTMobileViTV2Layer(nn.Module):
    def __init__(self, source):
        super().__init__()
        self.patch_height = source.patch_height
        self.patch_width = source.patch_width
        self.downsampling_layer = source.downsampling_layer
        self.conv_kxk = source.conv_kxk
        self.conv_1x1 = source.conv_1x1
        self.transformer = source.transformer
        self.layernorm = source.layernorm
        self.conv_projection = source.conv_projection

    def unfolding(self, feature_map):
        batch_size, channels, height, width = feature_map.shape
        blocks_height = height // self.patch_height
        blocks_width = width // self.patch_width
        patches = feature_map.reshape(
            batch_size,
            channels,
            blocks_height,
            self.patch_height,
            blocks_width,
            self.patch_width,
        )
        patches = patches.permute(0, 1, 3, 5, 2, 4)
        patches = patches.reshape(
            batch_size,
            channels,
            self.patch_height * self.patch_width,
            blocks_height * blocks_width,
        )
        return patches, (height, width)

    def folding(self, patches, output_size):
        batch_size, channels, _, _ = patches.shape
        height, width = output_size
        blocks_height = height // self.patch_height
        blocks_width = width // self.patch_width
        feature_map = patches.reshape(
            batch_size,
            channels,
            self.patch_height,
            self.patch_width,
            blocks_height,
            blocks_width,
        )
        feature_map = feature_map.permute(0, 1, 4, 2, 5, 3)
        return feature_map.reshape(batch_size, channels, height, width)

    def forward(self, features):
        if self.downsampling_layer is not None:
            features = self.downsampling_layer(features)
        features = self.conv_kxk(features)
        features = self.conv_1x1(features)
        patches, output_size = self.unfolding(features)
        patches = self.transformer(patches)
        patches = self.layernorm(patches)
        features = self.folding(patches, output_size)
        return self.conv_projection(features)


class LiteRTLinearSelfAttention(nn.Module):
    def __init__(self, source):
        super().__init__()
        self.qkv_proj = source.qkv_proj
        self.attn_dropout = source.attn_dropout
        self.out_proj = source.out_proj
        self.embed_dim = source.embed_dim

    def forward(self, hidden_states):
        qkv = self.qkv_proj(hidden_states)
        query, key, value = torch.split(
            qkv,
            [1, self.embed_dim, self.embed_dim],
            dim=1,
        )
        context_scores = torch.nn.functional.softmax(query, dim=-1)
        context_scores = self.attn_dropout(context_scores)
        key_by_patch = key.permute(0, 2, 1, 3)
        scores_by_patch = context_scores.permute(0, 2, 3, 1)
        context_by_patch = torch.matmul(key_by_patch, scores_by_patch)
        value_by_patch = torch.nn.functional.relu(value).permute(0, 2, 1, 3)
        scaled_value = torch.matmul(
            context_by_patch.unsqueeze(-1),
            value_by_patch.unsqueeze(-2),
        ).squeeze(-2)
        output = scaled_value.permute(0, 2, 1, 3)
        return self.out_proj(output)


def replace_export_modules(module):
    for name, child in list(module.named_children()):
        if isinstance(child, MobileViTV2LinearSelfAttention):
            child = LiteRTLinearSelfAttention(child)
            setattr(module, name, child)
        elif isinstance(child, MobileViTV2Layer):
            child = LiteRTMobileViTV2Layer(child)
            setattr(module, name, child)
        replace_export_modules(child)


sample_inputs = (test_dataset[0][0].unsqueeze(0),)

with torch.inference_mode():
    reference_output = model(*sample_inputs)

replace_export_modules(model)

with torch.inference_mode():
    export_output = model(*sample_inputs)

torch.testing.assert_close(export_output, reference_output, rtol=1e-5, atol=1e-6)
print(f"Entrada: {tuple(sample_inputs[0].shape)}")
print(f"Maior diferença: {(export_output - reference_output).abs().max().item():.3e}")

Entrada: (1, 3, 256, 256)
Maior diferença: 4.768e-07


## Exportação FP32

In [16]:
edge_model = litert_torch.convert(model, sample_inputs)
edge_model.export(str(FP32_PATH))

print(f"FP32: {FP32_PATH.stat().st_size / 1024 / 1024:.2f} MB")

(00:00) [START] LiteRT-Torch Convert

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default

(00:00) [START] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions

/opt/conda/lib/python3.11/copyreg.py:105: FutureWarning: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.
  return cls.__new__(cls, *args)


(00:02) [ DONE] LiteRT-Torch Convert > Torch Export: serving_default > ExportedProgram Run Decompositions (+00:01)

...

(00:11) [ DONE] LiteRT-Torch Convert (+00:11)

(00:00) [START] Write Model to tflite_models/model_fp32.tflite

I0000 00:00:1789815915.384851    4849 flatbuffer_export.cc:4342] Estimated count of arithmetic ops: 4.358 G  ops, equivalently 2.179 G  MACs


(00:00) [ DONE] Write Model to tflite_models/model_fp32.tflite (+00:00)

FP32: 17.42 MB


## Quantização

In [17]:
def create_calibration_data(model_path, dataset, num_samples, seed=42):
    interpreter = Interpreter(model_path=str(model_path))
    signatures = interpreter.get_signature_list()

    if len(signatures) != 1:
        raise ValueError(f"Expected one signature, found: {list(signatures)}")

    signature_name, signature = next(iter(signatures.items()))
    input_names = signature["inputs"]

    if len(input_names) != 1:
        raise ValueError(f"Expected one input, found: {input_names}")

    generator = torch.Generator().manual_seed(seed)
    sample_count = min(num_samples, len(dataset))
    indices = torch.randperm(len(dataset), generator=generator)[:sample_count].tolist()
    input_name = input_names[0]
    samples = [
        {input_name: dataset[index][0].unsqueeze(0).numpy().astype(np.float32)}
        for index in indices
    ]
    return {signature_name: samples}


def quantize_static(fp32_path, output_path, calibration_data, quantization_recipe):
    model_quantizer = quantizer.Quantizer(str(fp32_path))
    model_quantizer.load_quantization_recipe(quantization_recipe())
    calibration_result = model_quantizer.calibrate(calibration_data)
    quantized_model = model_quantizer.quantize(calibration_result)
    quantized_model.export_model(str(output_path), overwrite=True)
    print(f"{output_path.name}: {output_path.stat().st_size / 1024 / 1024:.2f} MB")

In [18]:
calibration_data = create_calibration_data(
    FP32_PATH,
    calibration_dataset,
    NUM_CALIB_SAMPLES,
)


In [19]:
quantize_static(FP32_PATH, INT8_PATH, calibration_data, recipe.static_wi8_ai8)

/opt/conda/lib/python3.11/site-packages/ai_edge_litert/interpreter.py:480: UserWarning: Warning: Enabling `experimental_preserve_all_tensors` with the BUILTIN or AUTO op resolver is intended for debugging purposes only. Be aware that this can significantly increase memory usage by storing all intermediate tensors. If you encounter memory problems or are not actively debugging, consider disabling this option.
  warnings.warn(
Generating Quantization Parameters::   0%|          | 0/789 [00:00<?, ?it/s] /opt/conda/lib/python3.11/site-packages/ai_edge_quantizer/algorithms/uniform_quantize/uniform_quantize_tensor.py:109: RuntimeWarning: invalid value encountered in cast
  return tensor.astype(qtype, copy=False)
Applying Transformations to tensors:: 100%|██████████| 1025/1025 [00:00<00:00, 29323.80it/s]


Model name: tflite_models/model_fp32.tflite
Original model size: 17.42 MiB
Quantized model size: 5.37 MiB
Quantization Ratio: 0.31 (3.2x smaller)
Total time: 199.79 ms
model_int8.tflite: 5.37 MB


In [20]:
quantize_static(FP32_PATH, INT8A16_PATH, calibration_data, recipe.static_wi8_ai16)


Applying Transformations to tensors:: 100%|██████████| 1025/1025 [00:00<00:00, 31504.92it/s]


Model name: tflite_models/model_fp32.tflite
Original model size: 17.42 MiB
Quantized model size: 5.45 MiB
Quantization Ratio: 0.31 (3.2x smaller)
Total time: 197.97 ms
model_int8a16.tflite: 5.45 MB


## Validação

In [21]:
def inspect_tflite(model_path):
    interpreter = Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    input_info = interpreter.get_input_details()[0]
    output_info = interpreter.get_output_details()[0]

    print(model_path.name)
    print(f"Input: {input_info['shape']} {input_info['dtype']} {input_info['quantization']}")
    print(f"Output: {output_info['shape']} {output_info['dtype']} {output_info['quantization']}")
    print(f"Size: {model_path.stat().st_size / 1024 / 1024:.2f} MB")


for model_path in (FP32_PATH, INT8A16_PATH, INT8_PATH):
    inspect_tflite(model_path)

model_fp32.tflite
Input: [  1   3 256 256] <class 'numpy.float32'> (0.0, 0)
Output: [ 1 20] <class 'numpy.float32'> (0.0, 0)
Size: 17.42 MB
model_int8a16.tflite
Input: [  1   3 256 256] <class 'numpy.int16'> (3.0361909011844546e-05, 0)
Output: [ 1 20] <class 'numpy.int16'> (0.00017463736003264785, 0)
Size: 5.45 MB
model_int8.tflite
Input: [  1   3 256 256] <class 'numpy.int8'> (0.003901445772498846, -128)
Output: [ 1 20] <class 'numpy.int8'> (0.027147473767399788, -84)
Size: 5.37 MB


In [22]:
def evaluate_tflite(model_path, data_loader, labels):
    interpreter = Interpreter(model_path=str(model_path))
    interpreter.allocate_tensors()
    input_details = interpreter.get_input_details()[0]
    output_details = interpreter.get_output_details()[0]
    input_scale, input_zero_point = input_details["quantization"]
    output_scale, output_zero_point = output_details["quantization"]
    true_labels = []
    predictions = []

    for images, batch_labels in data_loader:
        for image, label in zip(images.numpy(), batch_labels.numpy()):
            input_data = np.expand_dims(image, axis=0)

            if np.issubdtype(input_details["dtype"], np.integer):
                if input_scale == 0:
                    raise ValueError(f"Invalid input scale in {model_path}")
                limits = np.iinfo(input_details["dtype"])
                input_data = np.round(input_data / input_scale + input_zero_point)
                input_data = np.clip(input_data, limits.min, limits.max)

            input_data = input_data.astype(input_details["dtype"])
            interpreter.set_tensor(input_details["index"], input_data)
            interpreter.invoke()
            output = interpreter.get_tensor(output_details["index"])

            if np.issubdtype(output_details["dtype"], np.integer):
                output = (output.astype(np.float32) - output_zero_point) * output_scale

            true_labels.append(int(label))
            predictions.append(int(np.argmax(output, axis=1)[0]))

    accuracy = accuracy_score(true_labels, predictions)
    print(f"\n{model_path.name}: {accuracy:.4f}\n")
    print(
        classification_report(
            true_labels,
            predictions,
            labels=range(len(labels)),
            target_names=labels,
            digits=4,
            zero_division=0,
        )
    )
    return accuracy




In [28]:
results = {
    model_path.name: evaluate_tflite(model_path, test_loader, class_names)
    for model_path in (FP32_PATH, INT8_PATH)
}

/opt/conda/lib/python3.11/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()



model_fp32.tflite: 0.9769

                        precision    recall  f1-score   support

        Aglais_Urticae     0.9286    0.9630    0.9455        27
Anthocharis_Cardamines     0.9167    0.9778    0.9462        45
          Apatura_Iris     1.0000    1.0000    1.0000        35
       Argynnis_Paphia     1.0000    1.0000    1.0000        40
      Colias_Myrmidone     1.0000    0.9643    0.9818        28
     Gonepteryx_Rhamni     0.9200    1.0000    0.9583        23
            Inachis_Io     1.0000    1.0000    1.0000        28
 Iphiclides_Podalirius     1.0000    1.0000    1.0000        29
    Lycaena_Virgaureae     0.9412    0.9412    0.9412        17
            Lycaenidae     1.0000    1.0000    1.0000        41
       Maniola_Jurtina     1.0000    0.9756    0.9877        41
   Melanargia_Galathea     1.0000    1.0000    1.0000        41
     Nymphalis_Antiopa     0.9375    0.9375    0.9375        16
       Papilio_Machaon     1.0000    1.0000    1.0000        24
     Parnas

/opt/conda/lib/python3.11/multiprocessing/popen_fork.py:66: RuntimeWarning: os.fork() was called. os.fork() is incompatible with multithreaded code, and JAX is multithreaded, so this will likely lead to a deadlock.
  self.pid = os.fork()
